# KLA SEMICON 2026 — restoration training (Kaggle)

**Before running:**
1. Add the official archive as a Kaggle Dataset containing `train.zip`, then set `DATA_ZIP` below.
2. Settings → Accelerator → **GPU T4 x2** or **GPU P100**. We deliberately use a *single* GPU (see cell 2).
3. Settings → Internet **On** (needed for `git clone` and the optional `lpips` install).

Everything else is driven by `configs/base.yaml` — no hyperparameters are set in this notebook.

In [ ]:
# Get the code onto the machine. Two supported routes -- whichever is ready first:
#   A) clone from GitHub (set REPO_URL, needs Internet ON), or
#   B) add the repo folder to a Kaggle Dataset and point CODE_DATASET at it.
REPO_URL     = "https://github.com/USERNAME/kla-restore.git"   # route A
CODE_DATASET = "/kaggle/input/kla-code"                        # route B
DATA_ZIP     = "/kaggle/input/kla-semicon-2026/train.zip"      # <-- always set me
BUDGET_HOURS = 6.5

import os, shutil, subprocess
WORK = "/kaggle/working/kla-restore"

if not os.path.exists(WORK):
    if os.path.isdir(CODE_DATASET):
        # Kaggle input dirs are read-only; copy so training can write weights/results.
        shutil.copytree(CODE_DATASET, WORK)
        print("code from dataset:", CODE_DATASET)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
        print("code cloned from", REPO_URL)

os.chdir(WORK)
rev = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print("cwd:", os.getcwd(), "| git:", rev or "n/a")
assert os.path.exists(DATA_ZIP), f"DATA_ZIP not found: {DATA_ZIP}"
print("data zip:", DATA_ZIP, f"({os.path.getsize(DATA_ZIP)/1e6:.0f} MB)")

In [ ]:
# Pin to a single GPU. DataParallel across T4 x2 gives a modest speedup at the
# cost of a whole class of hangs/desyncs — not a risk worth taking on a run we
# only get to do once, overnight, against a deadline.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}, {p.total_memory/1e9:.1f} GB")

In [ ]:
!pip -q install lpips  # optional: only used for the final fine-tune + reporting

In [ ]:
# Pack the archive into contiguous memmaps (~2 min). Re-runnable if the
# session restarts — it skips the work if the output already exists.
import sys; sys.path.insert(0, ".")
from src.data import pack_from_zip
meta = pack_from_zip(DATA_ZIP, "/kaggle/working/packed")

In [ ]:
# Pre-flight gate: overfit 2 pairs. If this does not clear 35 dB the pipeline is
# broken and the long run would be wasted. ~1-2 minutes on a GPU.
# Do NOT pipe this through `tail` -- the exit code is the machine-readable result.
!python train.py --overfit 2 --steps 2000 --gate_db 35 --num_workers 2 --no_lpips --data_dir /kaggle/working/packed

In [ ]:
# Main run. train.py measures its own throughput over the first 200 steps and
# sizes the cosine schedule to fit BUDGET_HOURS, so the LR always lands at its
# minimum exactly when the budget runs out.
# Checkpoints + validation every 5k steps -> weights/best.pt survives a crash.
!python train.py --config configs/base.yaml \
    --data_dir /kaggle/working/packed \
    --out_dir /kaggle/working/weights \
    --hours {BUDGET_HOURS} \
    --num_workers 2

In [ ]:
!python evaluate.py --weights /kaggle/working/weights/best.pt \
    --data_dir /kaggle/working/packed --out_dir /kaggle/working/results

In [ ]:
# End-to-end inference timing on the validation split, measured the same way
# KLA measures it: process startup through to the last file written.
import json, numpy as np, os, sys
sys.path.insert(0, ".")
from src.data import split_indices
meta = json.load(open("/kaggle/working/packed/meta.json"))
_, val_idx = split_indices(meta["n"])
lr = np.load("/kaggle/working/packed/lr.npy", mmap_mode="r")
os.makedirs("/kaggle/working/bench_in", exist_ok=True)
for i in val_idx:
    np.save(f"/kaggle/working/bench_in/{i:06d}.npy", np.asarray(lr[i]))
print(len(val_idx), "files staged")

In [ ]:
!time python inference.py --input_dir /kaggle/working/bench_in \
    --output_dir /kaggle/working/bench_out \
    --weights /kaggle/working/weights/best.pt

In [ ]:
# Bundle the checkpoint + logs for download.
!cd /kaggle/working && zip -r submission_artifacts.zip weights results -x '*.pyc'
print("download /kaggle/working/submission_artifacts.zip")